### ROLE
You are a Senior AI Solutions Architect. Your task is to write a complete, executable Python script using `langgraph` and `langchain_openai`.

### OBJECTIVE
Create a ReAct Agent that manages a simulation of an "Antique Book".
The Agent must not run on OpenAI servers; it must connect to a local **vLLM** server running the Qwen 3 model.

### SPECIFICATIONS
1.  **LLM Backend:**
    * Model Name: "Qwen/Qwen3-30B-A3B-Instruct-2507-FP8"
    * Base URL: "http://localhost:8000/v1"
    * API Key: "EMPTY"
    * Temperature: 0.1 (Strict)

2.  **The "Antique Book" Logic (State):**
    * Create a class `AntiqueBook` or a global dictionary to hold state.
    * Attributes: `title` (str), `dust_level` (int, 0-10), `is_open` (bool), `pages` (dict of content).
    * Initial State: The book starts with `dust_level=10` (Very dusty) and `is_open=False`.

3.  **The Tools (LangChain @tool):**
    You must implement these functions and decorate them with `@tool` so the Agent can call them:
    * `inspect_book()`: Returns the current condition (dust level, open/closed status).
    * `clean_book()`: Sets `dust_level` to 0. Returns a success message.
    * `open_book()`: Sets `is_open` to True.
        * *Constraint:* If `dust_level` > 0, this tool must return an error string: "[ERROR] The book is too dusty to open. Clean it first."
    * `read_page(page_num: int)`: Returns text for the page.
        * *Constraint:* If `is_open` is False, return error: "[ERROR] The book is closed."

4.  **The Agent Structure:**
    * Use `create_react_agent` from `langgraph.prebuilt`.
    * Pass the custom `tools` list and the configured `ChatOpenAI` (vLLM) model.

5.  **Execution Simulation:**
    * In the `if __name__ == "__main__":` block, run a loop asking the agent to "Read page 1".
    * The Agent should self-correct: Try to open -> Fail (Dusty) -> Decide to Clean -> Open -> Read.

### STRICT CODING CONVENTIONS
* **NO EMOJIS:** Do not use emojis in any print statements or logs. Use text tags like [INFO], [ERROR], [TOOL].
* **Language:** English only.
* **Documentation:** Google-style docstrings for all functions.
* **Type Hinting:** Strict python typing (`List`, `Dict`, `int`, etc.).
* **Imports:** Ensure all imports (`langchain_openai`, `langgraph`, `pydantic`) are at the top.

### OUTPUT
Provide the full Python code in a single Markdown code block.

In [ ]:
# 1. Install & Setup Environment
import sys
import os

print("🔄 Setting up Environment...")

# Uninstall TensorFlow to prevent potential Protobuf conflicts (common in some Colab/Linux envs)
!{sys.executable} -m pip uninstall -y tensorflow

# Install vLLM (Server) and LangChain/LangGraph (Agent)
# Note: vLLM is officially supported on Linux. On Windows, this might require WSL2.
!{sys.executable} -m pip install -q vllm langgraph langchain-openai langchain-core nest_asyncio psutil

# Apply nest_asyncio to allow nested event loops (required for LangGraph in Notebooks)
import nest_asyncio
nest_asyncio.apply()

print("✅ Dependencies Installed & Asyncio Patched.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.8/84.8 kB 1.8 MB/s eta 0:00:00
[SYSTEM] Checking for vLLM server at http://localhost:8000/v1...
............................................................
[CRITICAL ERROR] vLLM server failed to start within timeout. Please ensure the vLLM startup cell is running.
[SYSTEM] Initializing ReAct Agent connected to local vLLM...


/tmp/ipython-input-1304339275.py:142: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent_executor = create_react_agent(llm, tools)


[USER] Read page 1
[SYSTEM ERROR] Simulation failed. Details: Connection error.


In [ ]:
# 2. Launch vLLM Server (Background Process)
import subprocess
import time
import requests
import os
import sys
import psutil

# --- CONFIGURATION ---
# Using Qwen 2.5 7B Instruct (Better balance of performance/memory than 30B)
MODEL_ID = "Qwen/Qwen2.5-7B-Instruct" 
PORT = 8000
BASE_URL = f"http://localhost:{PORT}/v1"

# --- CLEANUP PREVIOUS SERVER ---
# Cross-platform port cleanup
print(f"🧹 Checking port {PORT}...")
for proc in psutil.process_iter(['pid', 'name']):
    try:
        for conn in proc.connections(kind='inet'):
            if conn.laddr.port == PORT:
                print(f"   Killing process {proc.name()} (PID: {proc.pid}) on port {PORT}")
                proc.kill()
    except (psutil.NoSuchProcess, psutil.AccessDenied, psutil.ZombieProcess):
        pass
time.sleep(2)

# --- START SERVER ---
# Command to start vLLM in an OpenAI-compatible API mode
# We allow vLLM to auto-detect the best tool calling parser. 
# 'hermes' is specific to Hermes models; Qwen usually works best with default or 'qwen' if supported.
cmd = [
    sys.executable, "-m", "vllm.entrypoints.openai.api_server",
    "--model", MODEL_ID,
    "--trust-remote-code",
    "--dtype", "auto",
    "--max-model-len", "8192",
    "--gpu-memory-utilization", "0.90", # 90% VRAM Usage
    "--port", str(PORT),
    "--host", "0.0.0.0",
    "--enable-auto-tool-choice",      # Enable tool calling support
]

print(f"🚀 Starting vLLM Server for model: {MODEL_ID}...")
print(f"   Command: {' '.join(cmd)}")

# Redirect stdout/stderr to files 
with open("vllm_server.log", "w") as f:
    process = subprocess.Popen(cmd, stdout=f, stderr=subprocess.STDOUT)

# --- HEALTH CHECK ---
print("⏳ Waiting for server to come online (may take 2-5 mins)...")
print("   (Check 'vllm_server.log' if this hangs)")

start_time = time.time()
while True:
    # 1. Check if process died
    if process.poll() is not None:
        print("\n❌ Server process terminated unexpectedly.")
        try:
            with open("vllm_server.log", "r") as f:
                print(f"--- LOG TAIL ---\n{f.read()[-500:]}")
        except: pass
        break

    # 2. Check API Health
    try:
        if requests.get(f"{BASE_URL}/models").status_code == 200:
            print(f"\n✅ vLLM Server is Online @ {BASE_URL}")
            break
    except requests.exceptions.ConnectionError:
        pass

    # 3. Timeout (10 mins)
    if time.time() - start_time > 600:
        print("\n❌ Server failed to start within 10 minutes.")
        process.kill()
        break

    time.sleep(5)
    print(".", end="", flush=True)

✅ Tools Initialized.


In [ ]:
# 3. Define Logic: "Antique Book" Simulation & Tools
from typing import Dict, List, Optional
from langchain_core.tools import tool

# --- STATE MANAGEMENT ---
class AntiqueBook:
    """Represents the state of the antique book simulation."""
    def __init__(self) -> None:
        self.title: str = "The Alchemist's Journal"
        self.dust_level: int = 10  # Initial state: Very dusty
        self.is_open: bool = False
        self.pages: Dict[int, str] = {
            1: "The secret of the universe lies in the silence between thoughts.",
            2: "To transform lead into gold, one must first transform the soul.",
            3: "Beware the shadows that linger when the light is too bright."
        }
        print(f"📘 Book '{self.title}' initialized. Dust Level: {self.dust_level}/10.")

# Global state instance (Resetting state for fresh run)
book_state = AntiqueBook()

# --- TOOLS ---
@tool
def inspect_book() -> str:
    """Inspects the current condition of the book. Returns dust level and open/closed status."""
    status = "open" if book_state.is_open else "closed"
    return f"[INFO] The book '{book_state.title}' is currently {status}. Dust level: {book_state.dust_level}/10."

@tool
def clean_book() -> str:
    """Cleans the book to remove all dust. Use this if the book is too dusty to open."""
    book_state.dust_level = 0
    return "[INFO] You have carefully wiped the dust off the cover. Dust level is now 0."

@tool
def open_book() -> str:
    """Attempts to open the book. Fails if the book is dusty."""
    if book_state.dust_level > 0:
        return "[ERROR] The book is too dusty to open. Clean it first."
    book_state.is_open = True
    return "[INFO] You have successfully opened the book."

@tool
def read_page(page_num: int) -> str:
    """Reads the content of a specific page number. Structure: page_num (int)."""
    if not book_state.is_open:
        return "[ERROR] The book is closed. You cannot read it."
    
    content = book_state.pages.get(page_num)
    if content:
        return f"[PAGE {page_num}] {content}"
    else:
        return f"[ERROR] Page {page_num} does not exist."

print("✅ Tools & State Ready.")

✅ Server Check Helper Ready.


In [ ]:
# 4. Agent Execution (ReAct)
from langchain_openai import ChatOpenAI
from langgraph.prebuilt import create_react_agent
from langchain_core.messages import BaseMessage

print("Configuration:")
print(f"- Model: {MODEL_ID}")
print(f"- Endpoint: {BASE_URL}")

# Configure the LLM to point to our local vLLM server
# Note: 'api_key' is required by the client but ignored by local vLLM.
llm = ChatOpenAI(
    model=MODEL_ID,
    base_url=BASE_URL,
    api_key="EMPTY",
    temperature=0.1 # Low temperature for more deterministic tool usage
)

# List of tools we created
tools = [inspect_book, clean_book, open_book, read_page]

# Create the Agent
agent_executor = create_react_agent(llm, tools)

# --- SIMULATION ---
print("\n🤖 Agent Initialized. Starting Simulation...")
user_query = "Read page 1 of the book."
print(f"\n[USER REQUEST] {user_query}\n" + "="*50)

try:
    # Stream the execution to see the thought process (ReAct loop)
    # stream_mode="values" gives us the full state of messages at each step
    events = agent_executor.stream(
        {"messages": [("user", user_query)]},
        stream_mode="values"
    )

    for event in events:
        if "messages" in event:
            message = event["messages"][-1]
            
            # --- PRETTY PRINTING ---
            
            # 1. AI Message (Reasoning or Tool Call Decision)
            if message.type == "ai":
                if hasattr(message, "tool_calls") and message.tool_calls:
                    for tc in message.tool_calls:
                        print(f"👉 [AGENT DECISION] The agent decided to call tool: '{tc['name']}'")
                        print(f"    Arguments: {tc['args']}")
                elif message.content:
                    # Often the agent "thinks" before calling tools (Chain of Thought), or gives final answer
                    print(f"🧠 [AGENT THOUGHT/MSG] {message.content}")
            
            # 2. Tool Message (Result of the tool execution)
            elif message.type == "tool":
                print(f"🛠️ [TOOL OUTPUT] {message.content}")
                print("-" * 30)

except Exception as e:
    print(f"\n[ERROR] Simulation halted: {e}")
    print("Tip: If connection failed, ensure Cell 2 (Server) is running and 'Models' endpoint returns 200.")

[SYSTEM] Checking for vLLM server at http://localhost:8000/v1...
............................................................
[CRITICAL ERROR] vLLM server failed to start within timeout. Please ensure the vLLM startup cell is running.
